# 🧠 Neuro-Virality Predictor (fMRI Hook Attention AI)

Preveja a **ativação cerebral e a curva de retenção de ganchos de vídeo** usando o modelo de NeuroAI da Meta (**TRIBE v2**).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kethysb/neuro-virality-predictor/blob/main/notebooks/neuro_virality_predictor.ipynb)

---
### 📌 O que este notebook faz:
1. Extrai áudio, transcrição temporal exata das palavras (WhisperX) e frames de vídeo (30fps).
2. Funde três modalidades neurais (**Áudio:** Wav2Vec-BERT, **Texto:** LLaMA-3.2-3B, **Vídeo:** V-JEPA 2 / DINOv2).
3. Prevê a resposta fMRI (Sinal BOLD) em **20.484 vértices corticais** do cérebro humano segundo a segundo.
4. Renderiza o mapa 3D do cérebro sincronizado com a forma de onda do áudio e os frames do vídeo.

## 1. Instalação do TRIBE v2 e Dependências

In [ ]:
# Instalação rápida via uv
!pip install -q uv
!uv pip install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
# Garantir compatibilidade com Python 3.13 no Colab
!pip install -q --upgrade --force-reinstall numpy


## 2. Carregar o Modelo TRIBE v2 e a Malha Cerebral 3D

In [ ]:
from pathlib import Path
import torch
from tribev2.demo_utils import TribeModel
from tribev2.plotting import PlotBrain

CACHE_FOLDER = Path("./cache")
CACHE_FOLDER.mkdir(parents=True, exist_ok=True)

print("🧠 Carregando pesos do modelo TRIBE v2 na GPU...")
model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
)
plotter = PlotBrain(mesh="fsaverage5")
print("✅ Modelo e malha cortical fsaverage5 prontos!")


## 3. Extrair Eventos Multimodais do Vídeo (WhisperX + MoviePy)
Faça upload do seu vídeo de gancho (até 15s-60s) ou use um arquivo `.mp4`.

In [ ]:
import os
import nltk

# Configuração de tokenizers para evitar erro de download no Colab
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
nltk.download("punkt_tab", download_dir="/root/nltk_data", quiet=True)
nltk.download("punkt", download_dir="/root/nltk_data", quiet=True)

# Altere aqui para o caminho do seu vídeo:
video_path = "/content/gancho_teste_15s.mp4"

if not Path(video_path).exists():
    print(f"⚠️ Arquivo {video_path} não encontrado! Faça o upload no painel de Arquivos do Colab.")
else:
    print(f"🎬 Extraindo áudio, fala (WhisperX) e frames de {video_path}...")
    df = model.get_events_dataframe(video_path=video_path)
    display(df.head(10)[["type", "start", "duration", "filepath", "text", "context"]])


## 4. Autenticação Hugging Face & Predição fMRI (com Bypass Unsloth LLaMA-3.2)
> **Dica:** O LLaMA 3.2-3B da Meta exige aprovação prévia no HuggingFace. Para evitar filas de espera, usamos o espelho público e idêntico da `unsloth/Llama-3.2-3B`.

In [ ]:
from huggingface_hub import login
import transformers, huggingface_hub

# Opcional: Insira seu token de leitura gratuito do HF (se desejar aumentar o rate-limit)
# login("hf_...")

# Redirecionamento automático de segurança para evitar erro 403 Forbidden no LLaMA 3.2
orig_model_fp = transformers.AutoModel.from_pretrained
orig_tok_fp = transformers.AutoTokenizer.from_pretrained
orig_cfg_fp = transformers.AutoConfig.from_pretrained
orig_hf_down = huggingface_hub.hf_hub_download

def redirect_llama(name):
    if isinstance(name, str) and "meta-llama/Llama-3.2-3B" in name:
        return "unsloth/Llama-3.2-3B"
    return name

transformers.AutoModel.from_pretrained = lambda name, *args, **kwargs: orig_model_fp(redirect_llama(name), *args, **kwargs)
transformers.AutoTokenizer.from_pretrained = lambda name, *args, **kwargs: orig_tok_fp(redirect_llama(name), *args, **kwargs)
transformers.AutoConfig.from_pretrained = lambda name, *args, **kwargs: orig_cfg_fp(redirect_llama(name), *args, **kwargs)
huggingface_hub.hf_hub_download = lambda repo_id, *args, **kwargs: orig_hf_down(redirect_llama(repo_id), *args, **kwargs)

if hasattr(model.data, "text_feature") and model.data.text_feature is not None:
    model.data.text_feature.model_name = "unsloth/Llama-3.2-3B"

print("⚡ Executando inferência neural do TRIBE v2 em 20.484 vértices corticais...")
preds, segments = model.predict(events=df)
print(f"✅ Predição concluída! Formato: {preds.shape} (timesteps, vértices corticais)")


## 5. Renderizar o Mapa Cerebral 3D Segundo a Segundo

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

n_timesteps = min(15, len(preds))

print(f"🎨 Renderizando os primeiros {n_timesteps} segundos no córtex 3D...")
fig = plotter.plot_timesteps(
    preds[:n_timesteps],
    segments=segments[:n_timesteps],
    cmap="fire",
    norm_percentile=99,
    vmin=0.6,
    alpha_cmap=(0, 0.2),
    show_stimuli=True,
)

output_png = "/content/gancho_cerebro_3d_15s.png"
fig.savefig(output_png, bbox_inches="tight", dpi=150)
print(f"✅ Imagem em alta resolução salva em: {output_png}")

# Baixar automaticamente para o computador
files.download(output_png)
plt.show()
